## feats.py — Phase A feature extraction

Turns cleaned posts into a per-user behavioural feature matrix.

- **`score_post`** — scores one post: lexicon densities (parasocial / financial / victim, per group) plus stylistic signals (caps, exclamation, first-person), normalised per word.
- **`user_features`** — aggregates a user's posts into one feature row: lexicon means + peaks, posting dynamics (rate, burstiness, late-night share, active-days), sentiment mean/volatility, and escalation (harm density rising over time). Rate features are normalised by each user's own active span, so the 2013–2026 range across subreddits doesn't distort comparisons.
- **`ihs` / `transform`** — inverse-hyperbolic-sine compression + min-max scaling to 0–100, producing the final matrix for clustering.

Output: `user_features.parquet` → feeds Phase B.

### Double checking if the DAPT Model still exists

In [4]:
import os
os.chdir("/root/tf-project/venv/Projects/BehaviouralAnalysis")
print("src exists:", os.path.exists("src"))
print("src contents:", os.listdir("src") if os.path.exists("src") else "NO src FOLDER")

src exists: True
src contents: ['lexicons.py']


## Setup

In [3]:
import os, sys, glob
import numpy as np, pandas as pd
from tqdm import tqdm

os.chdir("/root/tf-project/venv/Projects/BehaviouralAnalysis")   # project root — fixes path bug
sys.path.append("src")                                            # feats.py + lexicons.py
from feats import score_post, user_features, transform

CLEAN = "data/interim/clean"          # feature-eligible subs only (NOT clean_dapt/popculture)
PROC  = "data/processed"
os.makedirs(PROC, exist_ok=True)

MIN_POSTS = 10        # drop users below this
N_USERS   = 40_000    # sample size for the matrix

print("cwd:", os.getcwd())
print("clean files:", len(glob.glob(f"{CLEAN}/*.parquet")))

cwd: /root/tf-project/venv/Projects/BehaviouralAnalysis
clean files: 36


## Loading posts & prep

In [4]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
sia = SentimentIntensityAnalyzer()

files = sorted(glob.glob(f"{CLEAN}/*.parquet"))
frames = []
for p in tqdm(files, desc="load"):
    d = pd.read_parquet(p, columns=["uid", "subreddit", "created_utc", "text"])
    frames.append(d)
df = pd.concat(frames, ignore_index=True)

df["dt"] = pd.to_datetime(df.created_utc, unit="s", utc=True)
df["text"] = df.text.str.lower()      # lexicon matching is lowercase
print("posts:", len(df), "| users:", df.uid.nunique())

load:   0%|          | 0/36 [00:00<?, ?it/s]

load: 100%|██████████| 36/36 [00:02<00:00, 16.90it/s]


posts: 2777991 | users: 270109


## Sampling users (keeping full histories)

In [5]:
counts = df.groupby("uid").size()
eligible = counts[counts >= MIN_POSTS].index
keep = pd.Series(eligible).sample(min(N_USERS, len(eligible)), random_state=0)
df = df[df.uid.isin(set(keep))].copy()
print("sampled users:", df.uid.nunique(), "| posts:", len(df))

sampled users: 39396 | posts: 2262722


## Sentiment per post 

In [6]:
tqdm.pandas(desc="sentiment")
df["sent"] = df.text.progress_apply(lambda t: sia.polarity_scores(t)["compound"])

sentiment: 100%|██████████| 2262722/2262722 [06:58<00:00, 5411.21it/s] 


## Checkpoint and scoring posts against lexicons

In [7]:
tqdm.pandas(desc="lexicon scoring")
scores = df.text.progress_apply(score_post).apply(pd.Series)
df = pd.concat([df.reset_index(drop=True), scores.reset_index(drop=True)], axis=1)
df.to_parquet(f"{PROC}/posts_scored.parquet")     # checkpoint — resume point
print("scored columns:", [c for c in scores.columns])

lexicon scoring: 100%|██████████| 2262722/2262722 [03:28<00:00, 10840.43it/s]


scored columns: ['parasocial_attachment', 'parasocial_fusion', 'parasocial_collective', 'parasocial_ritual', 'financial_collecting', 'financial_spend', 'financial_strain', 'victim_attack', 'victim_invasion', 'victim_fanwar', 'victim_hostility', 'victim_targeted', 'caps', 'excl', 'first_person', 'nwords']


In [8]:
import os
p = "data/processed/posts_scored.parquet"
print("saved:", os.path.exists(p), os.path.getsize(p)//1024//1024, "MB" if os.path.exists(p) else "")

saved: True 378 MB


## Checking lexicons integrity is it okay or not

In [9]:
for grp in ["victim_fanwar", "parasocial_attachment", "financial_spend", "victim_invasion"]:
    print(f"\n=== top 5 posts for {grp} ===")
    for t in df.nlargest(5, grp).text:
        print("  ", t[:130])


=== top 5 posts for victim_fanwar ===
   most likely antis/akgaes reported them
   why is everything just fanwars fanwars fanwars to you people
   did something happened about fanwars?
   everyone should be boycotting him
   bangchan called out akgaes

[removed]

=== top 5 posts for parasocial_attachment ===
   ultimate bias and bias wrecker 😋
   the bias wreckers are bias wreckering.
   the multistan to end all multistans
   yeontan: the ultimate bias wrecker ^my^hearteu
   bias (seungmin) bias wrecker (lk)

=== top 5 posts for financial_spend ===
   *presales, this only applies to presale
   b&n restocked ot7 set, pre-orders available
   omgomgomgomgomg i wasn't ready for this
   wts be deluxe albums (sealed/unsealed with preorder gift)
   [wts][usa] unsealed bts proof album

=== top 5 posts for victim_invasion ===
   that would give sasaengs validation
   he training for them sasaengs
   oh geez, not the sasaengs…🫠😑
   any moment that involves sasaengs.
   parasocial fans and sasae

## User-level aggregation

Collapses the 2.26M scored posts into one row per user (~30 behavioural features each).

For each user, `user_features` computes:
- **Lexicon densities** — mean + peak intensity across parasocial / financial / victim groups
- **Stylistic signals** — caps, exclamation, first-person use
- **Posting dynamics** — post rate, inter-post gap variability, burstiness, late-night share, active-days fraction (all normalised by the user's own active span)
- **Sentiment** — mean and volatility
- **Escalation** — change in harm-relevant density between the first and second half of a user's history

Output: `user_behavioural.parquet` — the behavioural half of the feature matrix. The embedding half (Next Cell of code, GPU) is joined to this in Cell 8.

In [10]:
tqdm.pandas(desc="aggregating")
Ubeh = df.groupby("uid").progress_apply(user_features)
Ubeh.to_parquet(f"{PROC}/user_behavioural.parquet")
print("behavioural matrix:", Ubeh.shape)
Ubeh.head()

aggregating: 100%|██████████| 39396/39396 [05:46<00:00, 113.85it/s]

behavioural matrix: (39396, 33)


,parasocial_attachment_mean,parasocial_fusion_mean,parasocial_collective_mean,parasocial_ritual_mean,financial_collecting_mean,financial_spend_mean,financial_strain_mean,victim_attack_mean,victim_invasion_mean,victim_fanwar_mean,...,burst,latenight,active_days_frac,span_days,n_subs,sent_mean,sent_vol,parasocial_escalation,financial_escalation,victim_escalation
uid,,,,,,,,,,,,,,,,,,,,,
000176d1b1741fb8,0.193589,0.015099,0.000000,0.092519,0.000000,0.000000,0.0,0.000000,0.0,0.000000,...,0.111111,0.121951,0.039099,1509.0,3.0,0.334191,0.504052,-0.213332,0.000000,-0.059488
000217ef25da9a72,0.128205,0.000000,0.000000,0.106838,0.000000,0.000000,0.0,0.000000,0.0,0.000000,...,0.160000,0.153846,0.029968,634.0,5.0,0.123223,0.401012,-0.470085,0.000000,0.000000
0004d8de3b1f301d,0.131355,0.000000,0.015886,0.000000,0.000000,0.000000,0.0,0.000000,0.0,0.000000,...,0.333333,0.000000,0.050725,138.0,1.0,0.224170,0.439472,0.135626,0.000000,0.000000
000801094ad91b70,0.518017,0.000000,0.079365,0.000000,0.000000,0.000000,0.0,0.116402,0.0,0.009311,...,0.101695,0.200000,0.023077,1820.0,4.0,0.219218,0.521428,0.313234,0.000000,0.197171
000af91699685b21,0.288671,0.000000,0.000000,0.068729,0.392157,0.246914,0.0,0.000000,0.0,0.000000,...,0.000000,0.200000,0.049383,243.0,1.0,0.484413,0.485650,0.670124,0.206194,0.000000


### Sanity check on the matrix to stop cluster breakage

In [11]:
import pandas as pd
Ubeh = pd.read_parquet("data/processed/user_behavioural.parquet")

# 1. any all-zero or near-constant columns? (dead features)
nunique = Ubeh.nunique()
print("low-variance columns:", nunique[nunique < 5].index.tolist())

# 2. any columns entirely NaN?
print("all-NaN columns:", Ubeh.columns[Ubeh.isna().all()].tolist())

# 3. did escalation features populate?
esc = [c for c in Ubeh.columns if "escalation" in c]
print("escalation cols:", esc)
print(Ubeh[esc].describe().T[["mean","min","max"]] if esc else "none")

# 4. quick look at the harm-relevant features
harm = [c for c in Ubeh.columns if any(x in c for x in ["victim","financial","parasocial"])]
print("\nfeature ranges:")
print(Ubeh[harm].describe().T[["mean","50%","max"]])

low-variance columns: []
all-NaN columns: []
escalation cols: ['parasocial_escalation', 'financial_escalation', 'victim_escalation']
                           mean        min        max
parasocial_escalation -0.021139 -27.142857  16.000000
financial_escalation   0.014070 -13.636364  13.072982
victim_escalation      0.008530 -18.744366   7.856115

feature ranges:
                                mean       50%         max
parasocial_attachment_mean  0.460789  0.331552    8.736157
parasocial_fusion_mean      0.030989  0.000000    6.623590
parasocial_collective_mean  0.085504  0.000000    7.797919
parasocial_ritual_mean      0.128215  0.017672   13.571429
financial_collecting_mean   0.192032  0.000000    9.840044
financial_spend_mean        0.116718  0.000000   12.673160
financial_strain_mean       0.006470  0.000000    2.043751
victim_attack_mean          0.029436  0.000000    4.211268
victim_invasion_mean        0.008437  0.000000    2.272727
victim_fanwar_mean          0.027827  0.0000

### Correllation check
If it prints redundant pairs (likely n_posts/rate, or a lexicon _mean with its _peak), we trim before clustering. If clean, nothing will popup

In [12]:
import numpy as np, pandas as pd
Ubeh = pd.read_parquet("data/processed/user_behavioural.parquet")
corr = Ubeh.corr(numeric_only=True).abs()
pairs = [(corr.columns[i], corr.columns[j], corr.iloc[i,j])
         for i in range(len(corr)) for j in range(i+1, len(corr))
         if corr.iloc[i,j] > 0.85]
for a, b, c in sorted(pairs, key=lambda x: -x[2]):
    print(f"{c:.2f}  {a}  <->  {b}")

In [13]:
import torch
print("free VRAM:", (torch.cuda.get_device_properties(0).total_memory
      - torch.cuda.memory_allocated())/1e9, "GB")

free VRAM: 12.878086144 GB


In [14]:
import numpy as np, pandas as pd
Ubeh = pd.read_parquet("data/processed/user_behavioural.parquet")
corr = Ubeh.corr(numeric_only=True).abs()
pairs = [(corr.columns[i], corr.columns[j], corr.iloc[i,j])
         for i in range(len(corr)) for j in range(i+1, len(corr))
         if corr.iloc[i,j] > 0.85]
for a, b, c in sorted(pairs, key=lambda x: -x[2]):
    print(f"{c:.2f}  {a}  <->  {b}")
print("done" if not pairs else f"{len(pairs)} redundant pairs")

done


## Embedding features (GPU)

Adds semantic depth beyond the lexicon counts by using the DAPT model itself.

- Loads `dapt_final` and passes every post through it, mean-pooling the token embeddings into one 768-dim vector per post.
- Averages a user's post vectors into a single user embedding (their overall semantic signature).
- Reduces 768 → 20 dimensions with PCA so the embeddings don't dwarf the 33 behavioural features when clustering.

This is what lifts the pipeline from keyword-counting to behavioural modelling

Output: `user_embeddings.parquet` → joined with the behavioural block.

In [15]:
df = pd.read_parquet("data/processed/posts_scored.parquet")
print("posts:", len(df), "users:", df.uid.nunique())

posts: 2262722 users: 39396


In [16]:
import torch
import numpy as np, pandas as pd
from transformers import AutoTokenizer, AutoModel
from tqdm import tqdm
from sklearn.decomposition import PCA

tok = AutoTokenizer.from_pretrained("models/dapt_final")
enc = AutoModel.from_pretrained("models/dapt_final").cuda().eval()

@torch.no_grad()
def embed(texts, bs=64):
    out = []
    for i in range(0, len(texts), bs):
        b = tok(texts[i:i+bs], truncation=True, max_length=192,
                padding=True, return_tensors="pt").to("cuda")
        with torch.autocast("cuda", dtype=torch.bfloat16):
            h = enc(**b).last_hidden_state
        m = b["attention_mask"].unsqueeze(-1)
        out.append(((h*m).sum(1)/m.sum(1)).float().cpu().numpy())
    return np.vstack(out)

# per-user mean of post embeddings
emb_rows = []
for uid, g in tqdm(df.groupby("uid"), desc="embed"):
    e = embed(g.text.tolist())
    emb_rows.append((uid, e.mean(0)))

uids, vecs = zip(*emb_rows)
E = pd.DataFrame(np.vstack(vecs), index=uids)

# reduce 768 -> 20 dims so embeddings don't dwarf the 33 behavioural features
Ep = pd.DataFrame(PCA(n_components=20, random_state=0).fit_transform(E),
                  index=E.index, columns=[f"emb_pc{i}" for i in range(20)])
Ep.to_parquet("data/processed/user_embeddings.parquet")
print("embedding features:", Ep.shape)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaModel LOAD REPORT from: models/dapt_final
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
embed: 100%|██████████| 39396/39396 [36:52<00:00, 17.81it/s]  


embedding features: (39396, 20)


## Merge & normalise (Phase A output)

Combines the two feature blocks into the final user matrix.

- Joins the 33 behavioural features with the 20 embedding features (inner join on user).
- Behavioural block → IHS transform (compresses long tails) then min-max to 0–100.
- Embedding block → z-score then min-max to 0–100, so both blocks share a scale and neither dominates the clustering distance.

Output: `user_features.parquet` — 39,396 users × 53 features, all 0–100, no NaNs. **This is the Phase A deliverable and the input to Phase B clustering.**

In [18]:
import pandas as pd
import sys; sys.path.append("src")
from feats import transform

Ubeh = pd.read_parquet("data/processed/user_behavioural.parquet")
Ep   = pd.read_parquet("data/processed/user_embeddings.parquet")

U = Ubeh.join(Ep, how="inner")
print("combined raw:", U.shape)

beh_cols = Ubeh.columns
Xbeh = transform(U[beh_cols])                                    # IHS + 0-100
Xemb = (U[Ep.columns] - U[Ep.columns].mean()) / U[Ep.columns].std()   # z-score
Xemb = (Xemb - Xemb.min()) / (Xemb.max() - Xemb.min()) * 100          # then 0-100

X = Xbeh.join(Xemb)
X.to_parquet("data/processed/user_features.parquet")
print("FINAL matrix:", X.shape)                                  # ~(39396, 53)
print("range check:", X.min().min(), "-", X.max().max())         # should be 0 - 100
print("NaNs:", X.isna().sum().sum())                             # should be 0

combined raw: (39396, 53)
FINAL matrix: (39396, 53)
range check: 0.0 - 100.0
NaNs: 0


## Phase A — complete
- 39,396 users (≥10 posts, sampled, full histories retained)
- 53 features: 33 behavioural (lexicon densities, posting dynamics, sentiment, escalation) + 20 DAPT embedding PCs
- All IHS-transformed and normalised to 0–100
- Output: user_features.parquet → feeds Phase B